# 1.6 Filer

**Kompetansemål:**
- gjøre rede for standarder for lagring, utveksling og sikring av ulike typer data
- bruke programmering til å innhente, analysere og presentere informasjon fra reelle datasett

## Grunnleggende om filer
Elektroniske dokumenter, digitale bilder, dataprogrammer, osv. lagres i **filer** på en datamaskin. De primære lagringsenhetene er platelager som HDD eller SSD, men data kan også lagres eksternt på minnepinner eller minnekort. Hvor og hvordan vi lagrer data innvirker på sikringen av dem. Vi skiller gjerne mellom programfiler og datafiler basert på innholdet selv om de er tilsynelatende like. Alle filer består av 0-er og 1-ere *(bit = binary digit)*. Disse er gruppert i **byte**, som består av 8 bit. Operativsystemet holder styr på hvor filen begynner (den første byten) og hvor filen slutter (den siste byten).

Filer har filnavn som består av `<navn>.<filtype>`. Både `<navn>` og `<filtype>` kan vi velge selv. Noen spesialtegn er ikke tillatt i navnene. Selv om vi står fritt til å velge navn på filtypen, bør vi følge vanlig standard: `.app` for applikasjonspakker i macOS, `.exe` eller `.com` for programfiler i Windows, `docx` for Word filer, `jpg` for bildefiler (av den typen), `html` for websider, osv. Se eksempelvis [fileinfo.com](https://fileinfo.com). Operativsystemet forbinder filtypene med bestemte programmer, men brukere kan også endre disse standardinnstillingene. Hvis vi dobbeltklikker på en `docx`-fil, starter Word og leser inn datafilen. Hvis vi dobbeltklikker på en `exe`-fil, tror operativsystemet det er et program og prøver å kjøre det.

Mange filtyper har dessuten en filsignatur (magisk nummer) først i fila. Dette kan programmene bruke til å gjenkjenne formatet (eller filtypen) til innholdet. For eksempel begynner en `.jpg`-fil ofte med `FF D8`, mens `.mp3`-filer typisk starter med `49 44 33 [ID3]`. For å utforske slike signaturer, kan vi se på denne [listen](https://en.wikipedia.org/wiki/List_of_file_signatures) som gir en oversikt over ulike filsignaturer. Vi kan se **hex**-kodene til filens innhold i Terminalvinduet i VS Code med `format-hex`-kommandoen, forutsatt at vi bruker PowerShell som er standard. Det virker ikke med Ledetekst (cmd). Vi kan begrense oss til den første linjen med `format-hex <filnavn> | Select-Object -First 1`. I macOS kan vi bruke `hexdump -C -n N <filnavn>` hvor N er antall bytes vi ønsker å se.

Rene tekstfiler (som `.txt`, `.csv` og `.json`) mangler vanligvis filsignatur. Når slike tekstfiler inneholder strukturerte data, omtales de ofte som "flate filer". En linje kan sammenliknes med en rad eller post i en database. En `.txt`-fil kan ha kolonner eller felt på fastsatte posisjoner, mens en `.csv`-fil *(comma-separated values)* skiller verdiene med komma, og da behøver ikke feltene stå i samme posisjon. `.json`-formatet skal vi lære om senere.

I VS Code kan vi opprette en fil som vi kaller `navn.txt` og skrive fornavnet vårt på første linje og etternavnet på andre linje. Deretter lagrer vi endringene med *File Save* (eller Ctrl+S) og skriver `format-hex navn.txt` i terminalvinduet. Da får vi se **hex**-koden til tegnene, og kan legge merke til at det står `0D 0A` mellom første og andre linje. Dette ble satt inn da vi trykket Enter-tasten. `0D` er hex for kontrolltegnet CR *(Carriage Return)*. Uttrykket stammer fra gamle skrivemaskiner hvor vi måtte skyve vogna tilbake med en spak da vi kom til slutten av en linje. `0A` er hex for LF *(Line Feed)*. Spaken kunne også brukes til å rulle valsen en eller flere linjer fram. I macOS brukes kun LF (`0A`), uten CR (`0D`)

## Filtilgang
Når vi skal lese data fra en fil i Python, bruker vi den innebygde funksjonen `open()`. Selv om filen vi ønsker å åpne ligger i samme mappe som programmet vårt, kan vi støte på problemer. Dette kan spesielt være tilfelle hvis vi har åpnet en annen mappe i VS Code via *File > Open Folder…*. For å omgå dette problemet, kan vi enten bruke *File > Open Folder…* for å åpne mappen hvor programmet ligger, eller justere innstillingene i VS Code. Klikk på tannhjulet nederst til venstre, velg *Settings*, søk deretter etter **python.terminal** og huk av for *Execute in File Dir*.

For å være på den sikre siden og alltid kunne nå riktig fil, uansett hvilken mappe vi jobber fra, kan vi bruke `pathlib`. Ved å bruke `Path(__file__).resolve().parent` får vi stien til mappen hvor programmet ligger. Selv om `Path` tillater oss å skjøte stien og filnavnet med `/`, kan dette være forvirrende siden Windows benytter `\`. Derfor kan det være fornuftig å holde seg til metoden `joinpath`. Hvis vi har en datafil med filbanen `data/txt/filnavn.txt`, hvor `data`-mappen ligger side om side med `src`-mappen, kan vi nå datafilen fra programmet med
`Path(__file__).resolve().parent.parent.joinpath('data','txt', 'filnavn.txt')`.

In [2]:
from pathlib import Path

print(f'Mappesti til programmet: {Path(__file__).resolve().parent}')
fil = Path(__file__).resolve().parent.parent.joinpath("data", "txt", "filnavn.txt")
print(f'Filbane: {fil}')
with fil.open() as f:
    print(f.read())


NameError: name '__file__' is not defined

I Jupyter fungerer ikke `__file__` fordi koden kjøres interaktivt. Vi kan imidlertid bruke `Path(os.getcwd()).parent.joinpath("data", "txt", "filnavn.txt")` forutsatt at vi ikke endrer arbeidskatalogen inni notatblokken.

## Filoperasjoner
Vi kan få hele innholdet som tekst (`read`), hele innholdet som en liste (`readlines`) eller linje for linje (`readline`).

In [1]:
with open('navn.txt') as fil:
    data = fil.read()
    print('read():')
    print(data)

    fil.seek(1)
    data = fil.read(2)
    print('\nseek(1); read(2):')
    print(data)

    fil.seek(0)
    linjer = fil.readlines()
    print('\nreadlines():')
    print(linjer)

    fil.seek(0)
    print('\nreadline():')
    print(fil.readline().lower(), end='')
    print(fil.readline().lower())

    fil.seek(0)
    print('\nfor linje in fil:')
    for linje in fil:
        print(linje.upper(), end='')


FileNotFoundError: [Errno 2] No such file or directory: 'navn.txt'

Når vi bruker `with open()` i stedet for `fil=open()`, slipper vi å lukke filen med `fil.close()`. Det er kjekt dersom noe skulle feile så vi aldri kommer til `fil.close()`. Filpekeren, posisjonen vi leser fra, flytter seg etter hvert som vi leser data fra filen. `seek()` flytter filpekeren til oppgitt posisjon. `read(2)` leser 2 tegn. `print()` avsluttes som standard med ny linje. Det kan vi undertrykke med `end=''` siden teksten allerede inneholder `\n`.

Når vi skal skrive til en fil, må vi velge `mode='w'`. Hvis filen ikke finnes, blir den opprettet. Hvis den finnes, blir den overskrevet. Dersom vi ønsker å legge noe til en eksisterende fil, må vi bruke `mode='a'`. Hvis teksten inneholder norske bokstaver (æ, ø og å), må vi også bruke `encoding='utf-8'`.

In [ ]:
with open('norsk.txt', mode='w', encoding='utf-8') as fil:
    fil.write('1: EN\n2: TO\n3: TRE\n')
    fil.write('æ ø å Æ Ø Å')


Hvis vi sjekker en [utf-8 tabell](https://www.utf8-chartable.de/), kan vi se at den siste bokstaven Å er lagret som `C3 85` (hex) i to bytes i motsetning til den første bokstaven 1, `31` (hex), som er lagret i én byte.

## Ta med minst dette
Mye av informasjonen i denne bolken er av generell karakter og bør allerede være kjent. Mens detaljer som heksadesimale koder og filsignaturer kan være nyttige i spesielle sammenhenger, er det viktigere å ha en grunnleggende forståelse av vanlige fileoperasjoner som `open`, `read`, `readlines`, `readline` og `close`. Når vi jobber med norsk tekst som kan inneholde æ, ø og å, må vi åpne filen med riktig tegnsett, for eksempel ved å bruke `open()` med `encoding='utf-8'`. I Windows brukes CR+LF (`\r\n`) for linjeskift i tekstfiler, mens macOS og Linux bruker kun LF (`\n`). Dette kan resultere i forskjeller i hvordan filer leses og vises på tvers av disse plattformene. Senere skal vi forenkle filhåndteringen ved å bruke biblioteker som `csv`, `json` og ikke minst `pandas`. Uansett, må vi huske å lukke en fil etter vi har åpnet den. Hvis ikke, risikerer vi blant annet at endringer går tapt, at filen blir ødelagt, eller at vi ikke får tilgang til den igjen. Dette kan unngås ved å bruke `with`-blokker.

## Øvingsoppgaver

### 1.6.1
a) Ta utgangspunkt i oppgave 1.5.1. I stedet for å legge heltallene inn i en liste, skal de lagres på hver sin linje i en `.txt`-fil som heter `heltall.txt`. Legg inn mellom 10 og 15 verdier. Kall programfilen `skriv_til_fil.py`.

   Skriv `notepad heltall.txt` i Terminalvinduet i VS Code og sjekk innholdet i filen med Notisblokk-programmet.

In [ ]:
# Skriv koden din her


b) Bytt `heltall.txt` med en annen elev. Skriv et program som leser innholdet i filen og
   - Skriver ut hvor mange tall filen inneholder
   - Skriver ut summen av tallene
   - Skriver ut alle tallene på én linje.

   Kall programfilen `les_fra_fil.py`.

In [ ]:
# Skriv koden din her


### 1.6.2
a) De tre filene `fil_1.xxx`, `fil_2.xxx` og `fil_3.xxx` har hver sin ukjente filtype, som er enten `.txt` (tekstfil), `.jpg` (bildefil) eller `.mp3` (lydfil). Skriv et program som
   - Lar brukeren skrive inn et filnavn
   - Sjekker de første bytene i filen og skriver ut om filsignaturen er `.jpg`, `.mp3` eller ukjent.

   Kall programmet `sjekk_signatur.py` og finn filtypen til de tre utleverte filene. Husk å åpne filene i *binary mode* med `mode = 'rb'`

In [ ]:
# Skriv koden din her


b) Dobbelt-klikk på filene i filutforskeren.
   Hva skjer? Forklar.

   Gi filene nytt navn med riktig filtype. Dobbelt-klikk på filene i filutforskeren.
   Hva skjer? Forklar.

   Gi filene det gamle navnet med filtype xxx. Prøv å åpne filene i en nettleser.
   Hva skjer? Forklar.

   Alle filene inneholder et kodeord.
   Hvilken fil inneholder hvilket kodeord?